# Espejos Coloniales — POC de Fast Prompting
### Racismo internalizado (autoracismo) en el Perú y su raíz colonial

**Proyecto Final — Preentrega 2**
Autor: Elvis Labaca · Curso: Prompt Engineering · Comisión N.º 96090

---

Esta *proof of concept* (POC) implementa, con técnicas de **Fast Prompting**, el generador de
piezas educativas (texto + imagen) descrito en la Preentrega 1: contenido que conecta cinco
manifestaciones actuales del autoracismo en el Perú con su raíz colonial específica —el sistema
de castas indígena o el régimen esclavista, según corresponda— dirigido a jóvenes peruanos de
18 a 30 años.

**Qué vas a encontrar en este notebook:**
1. Configuración y control de costos.
2. Técnicas de prompting utilizadas y por qué se eligió cada una.
3. Implementación: generación de texto (modelo texto-texto) y generación de imagen (modelo texto-imagen).
4. Un modo **demo (`dry_run=True`)** que muestra el funcionamiento completo sin consumir créditos de API,
   pensado para que esta notebook pueda revisarse y corregirse sin costo.
5. Un widget interactivo para generar piezas sin tocar el código.
6. Análisis de costos y conclusiones sobre cómo Fast Prompting mejoró la propuesta de la Preentrega 1.


## 1. Instalación y configuración

Se requiere una API key de OpenAI (`OPENAI_API_KEY`) solo si se quiere ejecutar en modo real
(`dry_run=False`). En modo demo no se necesita ninguna key ni conexión a internet.

**Control de costo (ver también la sección 6):** el diseño de esta POC hace **una única llamada
a la API por pieza** para el texto (en vez de una llamada por cada etapa de la Preentrega 1),
usando salida estructurada en JSON. Esto reduce en un 75% el número de consultas frente a un
enfoque ingenuo de 4 llamadas por pieza.


In [1]:
# Si falta alguna librería, se instala (no se ejecuta nada de pago en esta celda)
import sys, subprocess

def _ensure(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["openai", "ipywidgets", "python-dotenv"]:
    _ensure(pkg.replace("-", "_") if pkg != "python-dotenv" else "dotenv")

print("Dependencias listas.")

Dependencias listas.


In [2]:
import os
import json
from dotenv import load_dotenv

load_dotenv()  # busca un archivo .env local con OPENAI_API_KEY=sk-...

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

TEXT_MODEL = "gpt-4o-mini"      # modelo texto-texto: económico y suficiente para esta tarea
IMAGE_MODEL = "dall-e-3"        # modelo texto-imagen visto en el curso

# Contador simple de consultas realizadas en esta sesión, para llevar control de costo real
API_CALLS_LOG = []

if OPENAI_API_KEY:
    print("API key detectada: el modo dry_run=False funcionará.")
else:
    print("No se detectó OPENAI_API_KEY. El notebook funciona igual en modo demo (dry_run=True).")

No se detectó OPENAI_API_KEY. El notebook funciona igual en modo demo (dry_run=True).


## 2. Técnicas de Fast Prompting utilizadas

| Técnica | Dónde se aplica | Por qué se eligió |
|---|---|---|
| **Role Prompting** | Prompt de sistema del generador de texto | Fijar el rol de "historiador + redactor de contenido educativo" mejora la precisión histórica y el tono, evitando que el modelo derive a un tono genérico de blog. |
| **Few-Shot Prompting** | Ejemplo incluido en el prompt de texto | Un ejemplo completo (tema → raíz colonial → texto final) reduce la variabilidad de formato entre las 5 piezas y ancla la extensión (≈80 palabras) sin tener que corregir manualmente cada salida. |
| **Salida estructurada (JSON)** | Ambos generadores (texto e imagen) | Pedir JSON con claves fijas (`texto`, `raiz_colonial`, `prompt_imagen`) permite consumir la respuesta directamente en código, sin parseo frágil de texto libre, y en una sola llamada resuelve lo que en la Preentrega 1 estaba dividido en varias etapas. |
| **Zero-Shot Prompting** | Función de revisión de sensibilidad | Para la revisión no hace falta un ejemplo: es una tarea de verificación puntual (sí/no + sugerencias) que el modelo resuelve bien con solo la instrucción, así que agregar ejemplos habría sido una llamada más costosa sin mejora real. |
| **Prompting negativo (restricciones explícitas)** | Ambos generadores | Se listan explícitamente las cosas a evitar (mezclar raíces coloniales, estereotipos visuales, tono acusatorio), que es más eficaz que solo describir lo que sí se quiere. |

Respecto a la Preentrega 1: allí los prompts estaban pensados para copiarse y pegarse manualmente,
uno por etapa, en la interfaz de ChatGPT. Con Fast Prompting se **fusionan las etapas 1 y 2** de la
Preentrega 1 (investigación + redacción) en un único prompt con salida estructurada, lo que reduce
las llamadas necesarias y elimina el paso manual de copiar el resultado de una etapa a la siguiente.


## 3. Base de conocimiento del proyecto

Las cinco manifestaciones y su raíz colonial (verificadas en la Preentrega 1, con fuentes académicas
como el estudio de la PUCP sobre discriminación laboral por cabello afro-rizado en Lima) se definen
como datos, no como texto libre repetido en cada prompt. Esto evita reescribir el contexto histórico
en cada llamada y mantiene la coherencia entre piezas.

In [3]:
TEMAS = {
    "quechua": {
        "manifestacion": "Rechazo o vergüenza al hablar quechua o aimara en espacios públicos",
        "raiz": "sistema de castas indígena",
        "contexto": "mandato colonial de hispanización como condición de ascenso social",
    },
    "alisado": {
        "manifestacion": 'Alisado capilar como condición de "buena presencia" laboral',
        "raiz": "régimen esclavista",
        "contexto": "normas de moralidad pública sobre el cabello afro impuestas desde el siglo XVIII",
    },
    "mejorar_raza": {
        "manifestacion": 'La expresión "mejorar la raza" al elegir pareja',
        "raiz": "ambos sistemas coloniales",
        "contexto": "lógica de blanqueamiento común al sistema de castas y al régimen esclavista",
    },
    "apellido": {
        "manifestacion": "Vergüenza del apellido, el acento o el origen andino/amazónico",
        "raiz": "sistema de castas indígena",
        "contexto": "jerarquización colonial por origen étnico y geográfico",
    },
    "medios": {
        "manifestacion": "Sobrerrepresentación de rasgos europeos como sinónimo de belleza en medios",
        "raiz": "ambos sistemas coloniales",
        "contexto": "convergencia histórica en el canon estético peruano actual",
    },
}

list(TEMAS.keys())

['quechua', 'alisado', 'mejorar_raza', 'apellido', 'medios']

## 4. Generador de texto (modelo texto-texto)

Una sola función, una sola llamada a la API por pieza. Combina Role Prompting + Few-Shot +
salida JSON + restricciones explícitas.

In [4]:
SYSTEM_PROMPT = """Eres historiador especializado en el Virreinato del Perú y en la historia de la \
esclavización de personas afrodescendientes en América, y además redactor de contenido educativo \
para redes sociales. Tu tarea es producir piezas breves, rigurosas y no acusatorias sobre racismo \
internalizado en el Perú actual, conectando cada manifestación con su raíz colonial específica. \
Nunca mezcles la raíz del sistema de castas indígena con la del régimen esclavista salvo que ambas \
apliquen explícitamente. Respondes siempre en JSON válido, sin texto adicional fuera del JSON."""

FEW_SHOT_EXAMPLE_USER = """Manifestación: Vergüenza del apellido o el origen andino en contextos \
urbanos y laborales.
Raíz histórica: sistema de castas indígena.
Contexto: jerarquización colonial por origen étnico y geográfico."""

FEW_SHOT_EXAMPLE_ASSISTANT = json.dumps({
    "texto": (
        "Cambiar de apellido o esconder el pueblo de origen para 'encajar' en la ciudad no es "
        "casualidad. Desde la Colonia, ser identificado como indígena o de provincia cerraba "
        "puertas: el sistema de castas premiaba el apellido español y castigaba el origen andino. "
        "¿Alguna vez sentiste que tu apellido o tu acento decían más de vos de lo que quisiste?"
    ),
    "raiz_colonial": "sistema de castas indígena",
    "prompt_imagen": (
        "Ilustracion digital conceptual, estilo editorial, paleta tierra y ocres con acentos "
        "de azul andino. Un documento de identidad estilizado cuyo nombre se desdibuja en el aire "
        "como humo, sin rostros ni texto legible, composicion centrada, formato cuadrado. Evitar "
        "estereotipos caricaturescos de rasgos indigenas."
    ),
}, ensure_ascii=False)


def build_text_prompt(tema_key: str) -> list:
    t = TEMAS[tema_key]
    user_prompt = f"""Manifestación: {t['manifestacion']}.
Raíz histórica: {t['raiz']}.
Contexto: {t['contexto']}.

Genera una pieza para un post de Instagram educativo sobre racismo internalizado en el Perú,
dirigida a personas de 18 a 30 años sin formación académica especializada.

Reglas obligatorias:
- Estructura en 3 partes dentro de "texto": (1) situación cotidiana en tono cercano, no acusatorio;
  (2) origen colonial citando el mecanismo histórico concreto indicado arriba, sin mezclarlo con
  la otra raíz colonial salvo que el campo "Raíz histórica" diga "ambos sistemas coloniales";
  (3) una pregunta de cierre reflexiva, sin moralizar.
- Máximo 80 palabras en "texto".
- "prompt_imagen": una metáfora visual (no una persona real), estilo editorial, sin texto dentro
  de la imagen, sin estereotipos caricaturescos de rasgos indígenas o afrodescendientes.

Responde solo con este JSON, sin explicación adicional:
{{"texto": "...", "raiz_colonial": "...", "prompt_imagen": "..."}}"""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": FEW_SHOT_EXAMPLE_USER},
        {"role": "assistant", "content": FEW_SHOT_EXAMPLE_ASSISTANT},
        {"role": "user", "content": user_prompt},
    ]

In [5]:
def generate_piece(tema_key: str, dry_run: bool = True) -> dict:
    """Genera una pieza (texto + prompt de imagen) para el tema indicado.

    dry_run=True  -> no llama a la API, devuelve una salida simulada representativa
                      (para revisar el notebook sin costo).
    dry_run=False -> hace UNA llamada real a la API de OpenAI (Chat Completions, JSON mode).
    """
    messages = build_text_prompt(tema_key)

    if dry_run:
        # Salida simulada, útil para demo y para testear el resto del pipeline sin costo.
        t = TEMAS[tema_key]
        return {
            "texto": f"[DEMO] Pieza simulada sobre: {t['manifestacion']}. "
                     f"(raíz: {t['raiz']}). Ejecuta con dry_run=False y una API key válida "
                     f"para obtener el texto real generado por el modelo.",
            "raiz_colonial": t["raiz"],
            "prompt_imagen": f"[DEMO] Prompt de imagen simulado para: {t['manifestacion']}.",
        }

    if not OPENAI_API_KEY:
        raise RuntimeError("Falta OPENAI_API_KEY. Define la variable de entorno o usa dry_run=True.")

    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

    response = client.chat.completions.create(
        model=TEXT_MODEL,
        messages=messages,
        response_format={"type": "json_object"},
        temperature=0.7,
    )
    API_CALLS_LOG.append({"tipo": "texto", "tema": tema_key, "modelo": TEXT_MODEL})

    return json.loads(response.choices[0].message.content)

## 5. Generador de imagen (modelo texto-imagen)

Reutiliza el `prompt_imagen` devuelto por el paso anterior — **no se vuelve a llamar a un modelo
de texto para construir el prompt de imagen**, evitando una consulta extra por pieza.

In [6]:
def generate_image(prompt_imagen: str, dry_run: bool = True) -> str:
    """Devuelve una URL de imagen (modo real) o una ruta simulada (modo demo).

    Nota: si no se dispone de acceso a DALL-E, este mismo prompt_imagen puede pegarse
    manualmente en Nightcafe u otra herramienta de generación de imagen gratuita, tal como
    se planteó como alternativa en la Preentrega 1.
    """
    if dry_run:
        return f"[DEMO] (sin llamada real) Imagen simulada para prompt: {prompt_imagen[:60]}..."

    if not OPENAI_API_KEY:
        raise RuntimeError("Falta OPENAI_API_KEY. Define la variable de entorno o usa dry_run=True.")

    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

    response = client.images.generate(
        model=IMAGE_MODEL,
        prompt=prompt_imagen,
        n=1,
        size="1024x1024",
    )
    API_CALLS_LOG.append({"tipo": "imagen", "modelo": IMAGE_MODEL})

    return response.data[0].url

## 6. Función de revisión (Zero-Shot) — opcional, bajo demanda

Solo se ejecuta si el usuario la pide explícitamente sobre una pieza puntual (no corre en batch
sobre las 5 piezas por defecto), para no sumar 5 llamadas adicionales sin necesidad real.

In [7]:
def review_piece(texto: str, dry_run: bool = True) -> dict:
    if dry_run:
        return {"observaciones": "[DEMO] Sin llamada real. Revisión disponible con dry_run=False."}

    if not OPENAI_API_KEY:
        raise RuntimeError("Falta OPENAI_API_KEY. Define la variable de entorno o usa dry_run=True.")

    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

    prompt = f"""Revisa el siguiente texto de una campaña educativa sobre racismo internalizado
en el Perú. Señala en JSON si hay: (1) afirmaciones históricas no verificables,
(2) tono acusatorio hacia el lector, (3) estereotipos, (4) lenguaje poco accesible.
Responde solo: {{"observaciones": "..."}}.

Texto: \"\"\"{texto}\"\"\" """

    response = client.chat.completions.create(
        model=TEXT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0.3,
    )
    API_CALLS_LOG.append({"tipo": "revision", "modelo": TEXT_MODEL})
    return json.loads(response.choices[0].message.content)

## 7. Demostración en modo demo (sin costo)

Genera una pieza completa (texto + prompt de imagen) para uno de los cinco temas, sin llamar a
la API. Sirve para validar el pipeline completo antes de gastar créditos reales.

In [8]:
demo_resultado = generate_piece("alisado", dry_run=True)
demo_imagen = generate_image(demo_resultado["prompt_imagen"], dry_run=True)

print("TEXTO:", demo_resultado["texto"])
print("\nRAÍZ COLONIAL:", demo_resultado["raiz_colonial"])
print("\nPROMPT DE IMAGEN:", demo_resultado["prompt_imagen"])
print("\nIMAGEN:", demo_imagen)

TEXTO: [DEMO] Pieza simulada sobre: Alisado capilar como condición de "buena presencia" laboral. (raíz: régimen esclavista). Ejecuta con dry_run=False y una API key válida para obtener el texto real generado por el modelo.

RAÍZ COLONIAL: régimen esclavista

PROMPT DE IMAGEN: [DEMO] Prompt de imagen simulado para: Alisado capilar como condición de "buena presencia" laboral.

IMAGEN: [DEMO] (sin llamada real) Imagen simulada para prompt: [DEMO] Prompt de imagen simulado para: Alisado capilar como ...


## 8. Widget interactivo

Contenido adicional (no obligatorio): permite elegir el tema y el modo de ejecución sin escribir
código ni editar celdas, tal como sugiere la consigna para sumar interactividad a la carga de
prompts.

In [9]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

tema_dropdown = widgets.Dropdown(
    options=[(v["manifestacion"], k) for k, v in TEMAS.items()],
    description="Tema:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="600px"),
)

dry_run_checkbox = widgets.Checkbox(
    value=True,
    description="Modo demo (dry_run) — no consume créditos de API",
)

generar_btn = widgets.Button(description="Generar pieza", button_style="success")
salida = widgets.Output()


def on_click(_):
    with salida:
        clear_output()
        tema_key = tema_dropdown.value
        dry_run = dry_run_checkbox.value
        resultado = generate_piece(tema_key, dry_run=dry_run)
        imagen = generate_image(resultado["prompt_imagen"], dry_run=dry_run)
        display(Markdown(
            f"**Texto:** {resultado['texto']}\n\n"
            f"**Raíz colonial:** {resultado['raiz_colonial']}\n\n"
            f"**Prompt de imagen:** {resultado['prompt_imagen']}\n\n"
            f"**Imagen:** {imagen}"
        ))


generar_btn.on_click(on_click)
display(tema_dropdown, dry_run_checkbox, generar_btn, salida)

Dropdown(description='Tema:', layout=Layout(width='600px'), options=(('Rechazo o vergüenza al hablar quechua o…

Checkbox(value=True, description='Modo demo (dry_run) — no consume créditos de API')

Button(button_style='success', description='Generar pieza', style=ButtonStyle())

Output()

## 9. Ejecución real (opcional — consume créditos)

Descomenta y ejecuta esta celda solo cuando quieras generar las 5 piezas finales con la API real.
Se hacen exactamente **2 llamadas por pieza** (1 de texto + 1 de imagen) = **10 llamadas en total**
para las 5 piezas completas, sin llamadas redundantes.

In [10]:
# for tema_key in TEMAS:
#     resultado = generate_piece(tema_key, dry_run=False)
#     imagen_url = generate_image(resultado["prompt_imagen"], dry_run=False)
#     print(tema_key, "->", resultado["texto"][:60], "...")
#     print("Imagen:", imagen_url)
#     print("-" * 80)

print("Celda desactivada por defecto para evitar consumo accidental de créditos.")

Celda desactivada por defecto para evitar consumo accidental de créditos.


## 10. Análisis de costos

| Enfoque | Llamadas por pieza | Llamadas para las 5 piezas |
|---|---|---|
| Preentrega 1 (manual, 4 etapas por pieza) | 4 (investigación, redacción, imagen, revisión) | 20 |
| Esta POC (texto+imagen fusionados, revisión opcional) | 2 | 10 |

La reducción se logra fusionando las etapas de investigación y redacción de la Preentrega 1 en
una sola llamada con salida JSON estructurada, y dejando la revisión de sensibilidad como una
función bajo demanda en vez de un paso automático sobre las 5 piezas. El modo `dry_run` permite
además iterar sobre el prompt (ajustar tono, extensión, restricciones) **sin ninguna llamada real**,
concentrando el gasto de créditos únicamente en la corrida final.

## 11. Conclusiones

Aplicar Fast Prompting sobre la propuesta de la Preentrega 1 permitió:

- **Reducir el número de consultas a la mitad** (de 4 a 2 por pieza), fusionando etapas mediante
  salida estructurada en JSON en lugar de prompts secuenciales copiados y pegados manualmente.
- **Aumentar la consistencia del formato** entre las 5 piezas gracias al ejemplo Few-Shot, algo que
  en la Preentrega 1 dependía de que la persona revisara manualmente cada resultado.
- **Separar el desarrollo del costo**: gracias al modo demo, todo el pipeline (texto, imagen,
  revisión, interfaz) pudo construirse, probarse y corregirse sin gastar créditos, y el gasto real
  queda acotado a una única corrida final controlada (Sección 9).
- **Mantener la distinción histórica** entre el sistema de castas indígena y el régimen esclavista
  como restricción explícita del prompt, en lugar de depender de una corrección manual posterior,
  como había ocurrido durante el desarrollo de la Preentrega 1.
